In [1]:
%pip install -q -U pypdf pandas numpy langchain-text-splitters sentence-transformers chromadb transformers accelerate torch

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import glob       # glob is used to find files that match a pattern.
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
from pypdf import PdfReader

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
    MarkdownTextSplitter,
)

import torch
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

In [2]:
DATA_DIR = Path("./data")
CHROMA_DIR = Path("./chroma_db")

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL_NAME = "google/flan-t5-base"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
TOP_K = 4

''' 
Chunk 1: 0 ─────────────── 800
         |<---- 800 ----->|

Chunk 2:              680 ─────────────── 1480
                      |<---- 800 ------->|

Chunk 3:                              1360 ─────────────── 2160
                                      |<---- 800 ------->|

'''

DATA_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Data:", DATA_DIR.resolve())
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Generation model:", GENERATION_MODEL_NAME)

Device: cpu
Data: /Users/macbook/Downloads/databrick-mini-course-end-to-end-project/Rag-project/data
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Generation model: google/flan-t5-base


In [3]:
# Fallback sample corpus: makes the notebook runnable immediately.
sample_docs = {
    "accessibility_manual.txt": """
Accessibility Support Guidelines

The ACCESSIBILITY-2024 device supports customers with visual, auditory, motor, and cognitive accessibility requirements.

Environment Requirements:
Maintain an operating temperature between 0°C and 40°C and humidity between 10% and 90% non-condensing.
The device can connect through Wi-Fi or Ethernet.

Performance Optimization:
Use wired Ethernet for critical services when possible. Optimize Wi-Fi channels to reduce interference,
use quality-of-service settings where appropriate, and limit unnecessary background network usage.

Security:
Use role-based access control and multi-factor authentication for administrator accounts. Review audit logs regularly.

Warranty:
The device is covered by a 24-month limited warranty for manufacturing defects and hardware failure under normal use.
Returns are accepted within 30 days with proof of purchase.
""",
    "account_management_manual.txt": """
Customer Account Management Guide

The account management system supports account creation, modification, authorized user management,
authentication, security controls, and audit logging.

Authentication:
Supported authentication methods include username/password, two-factor authentication, and digital certificates.

Security:
Administrators should use least-privilege access, review audit logs, and enable multi-factor authentication
for high-risk accounts.

Troubleshooting:
For login failures, verify credentials, check account status, review service availability, and inspect logs.
"""
}# removes unnecessary whitespace at the beginning and end .strip(), dedent() removes the unnecessary indentation.

if not list(DATA_DIR.glob("*.pdf")) and not list(DATA_DIR.glob("*.txt")):
    for filename, content in sample_docs.items():
        (DATA_DIR / filename).write_text(textwrap.dedent(content).strip(), encoding="utf-8")
    print("No documents found. Created sample documents.")
else:
    print("Existing documents found; sample documents were not created.")

print("\nFiles:")
for p in sorted(DATA_DIR.iterdir()):
    print(" -", p.name)

No documents found. Created sample documents.

Files:
 - accessibility_manual.txt
 - account_management_manual.txt


In [4]:
def load_documents(data_dir: Path):
    documents = []

    for pdf_path in sorted(data_dir.glob("*.pdf")):
        reader = PdfReader(str(pdf_path))

        for page_number, page in enumerate(reader.pages, start=1):
            text = (page.extract_text() or "").strip()

            if text:
                documents.append({
                    "doc_id": pdf_path.stem,
                    "source": pdf_path.name,
                    "page": page_number,
                    "content": text,
                })

    # TXT support is useful for a zero-setup training demo.
    for txt_path in sorted(data_dir.glob("*.txt")):
        text = txt_path.read_text(encoding="utf-8", errors="ignore").strip()

        if text:
            documents.append({
                "doc_id": txt_path.stem,
                "source": txt_path.name,
                "page": None,
                "content": text,
            })

    return documents


documents = load_documents(DATA_DIR) # DATA_DIR = Path("./data") "Load all the documents from the data directory."
documents_df = pd.DataFrame(documents)

print("Documents/pages loaded:", len(documents_df))
display(documents_df.head())

Documents/pages loaded: 2


,doc_id,source,page,content
0,accessibility_manual,accessibility_manual.txt,None,Accessibility Support Guidelines\n\nThe ACCESS...
1,account_management_manual,account_management_manual.txt,None,Customer Account Management Guide\n\nThe accou...


In [5]:
# Main demo: RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for doc in documents:
    for chunk_index, chunk in enumerate(splitter.split_text(doc["content"])):
        chunks.append({
            "chunk_id": f'{doc["doc_id"]}_{doc["page"]}_{chunk_index}',
            "doc_id": doc["doc_id"],
            "source": doc["source"],
            "page": doc["page"],
            "chunk_index": chunk_index,
            "content": chunk,
        })

chunks_df = pd.DataFrame(chunks)

print("Total chunks:", len(chunks_df))
display(chunks_df.head())



Total chunks: 3


,chunk_id,doc_id,source,page,chunk_index,content
0,accessibility_manual_None_0,accessibility_manual,accessibility_manual.txt,None,0,Accessibility Support Guidelines\n\nThe ACCESS...
1,accessibility_manual_None_1,accessibility_manual,accessibility_manual.txt,None,1,Warranty:\nThe device is covered by a 24-month...
2,account_management_manual_None_0,account_management_manual,account_management_manual.txt,None,0,Customer Account Management Guide\n\nThe accou...


In [6]:
for i, row in chunks_df.head(5).iterrows():
    print(f"--- Chunk {i} | {row['source']} | page={row['page']} ---")
    print(row["content"][:700])
    print()

# Take the first 5 chunks, and for each one show me its index, source file, page number, and first 700 characters of its text.

--- Chunk 0 | accessibility_manual.txt | page=None ---
Accessibility Support Guidelines

The ACCESSIBILITY-2024 device supports customers with visual, auditory, motor, and cognitive accessibility requirements.

Environment Requirements:
Maintain an operating temperature between 0°C and 40°C and humidity between 10% and 90% non-condensing.
The device can connect through Wi-Fi or Ethernet.

Performance Optimization:
Use wired Ethernet for critical services when possible. Optimize Wi-Fi channels to reduce interference,
use quality-of-service settings where appropriate, and limit unnecessary background network usage.

Security:
Use role-based access control and multi-factor authentication for administrator accounts. Review audit logs regularly.

--- Chunk 1 | accessibility_manual.txt | page=None ---
Warranty:
The device is covered by a 24-month limited warranty for manufacturing defects and hardware failure under normal use.
Returns are accepted within 30 days with proof of purchase.

--- 

In [7]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

chunk_texts = chunks_df["content"].tolist()

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32, # Process up to 32 chunks at a time.
    show_progress_bar=True,
    normalize_embeddings=True
)

chunk_embeddings = np.asarray(chunk_embeddings)

print("Embedding matrix shape:", chunk_embeddings.shape)
print("Embedding dimension:", chunk_embeddings.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (3, 384)
Embedding dimension: 384


In [8]:
# Create a persistent local Chroma database.
# No external server is required for this demo.

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collection_name = "rag_training_demo"

try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"description": "Open-source RAG training collection"}
)

collection.add(
    ids=chunks_df["chunk_id"].tolist(),
    documents=chunks_df["content"].tolist(),
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunks_df[["doc_id", "source", "page", "chunk_index"]]
        .fillna("")
        .to_dict(orient="records") 
)

print("Vectors stored:", collection.count())

Vectors stored: 3


In [9]:
def retrieve(question: str, top_k: int = TOP_K):
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    ) #ChromaDB, take my question's vector, compare it against all the vectors you have stored, and give me the top K most similar chunks, along with their text, metadata, and distances.

    rows = []

    for i in range(len(results["documents"][0])): # [0] is inside the list 
        rows.append({
            "rank": i + 1,
            "content": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })

    return pd.DataFrame(rows)


question = "How long is the warranty?"
retrieved_df = retrieve(question, top_k=TOP_K)

display(retrieved_df[["rank", "distance", "metadata", "content"]])

,rank,distance,metadata,content
0,1,0.834513,"{'source': 'accessibility_manual.txt', 'doc_id...",Warranty:\nThe device is covered by a 24-month...
1,2,1.813772,"{'source': 'accessibility_manual.txt', 'page':...",Accessibility Support Guidelines\n\nThe ACCESS...
2,3,2.004651,"{'page': '', 'source': 'account_management_man...",Customer Account Management Guide\n\nThe accou...


In [10]:
def build_context(retrieved_df: pd.DataFrame) -> str:
    context_parts = []

    for _, row in retrieved_df.iterrows():
        metadata = row["metadata"]

        page_info = (
            f", page {metadata['page']}"
            if metadata.get("page") not in ("", None)
            else ""
        )

        context_parts.append(
            f"[Source: {metadata['source']}{page_info}]\n"
            f"{row['content']}"
        )

    return "\n\n".join(context_parts)


context = build_context(retrieved_df)
print(context)

[Source: accessibility_manual.txt]
Warranty:
The device is covered by a 24-month limited warranty for manufacturing defects and hardware failure under normal use.
Returns are accepted within 30 days with proof of purchase.

[Source: accessibility_manual.txt]
Accessibility Support Guidelines

The ACCESSIBILITY-2024 device supports customers with visual, auditory, motor, and cognitive accessibility requirements.

Environment Requirements:
Maintain an operating temperature between 0°C and 40°C and humidity between 10% and 90% non-condensing.
The device can connect through Wi-Fi or Ethernet.

Performance Optimization:
Use wired Ethernet for critical services when possible. Optimize Wi-Fi channels to reduce interference,
use quality-of-service settings where appropriate, and limit unnecessary background network usage.

Security:
Use role-based access control and multi-factor authentication for administrator accounts. Review audit logs regularly.

[Source: account_management_manual.txt]
Cust

In [11]:
def build_rag_prompt(question: str, context: str) -> str:
    return f"""
Answer the question using ONLY the context provided below.

Rules:
- Do not invent facts.
- If the answer is not present in the context, say:
  "I could not find this information in the provided documents."
- Keep the answer concise and factual.
- Mention the relevant source when possible.

Context:
{context}

Question:
{question}

Answer:
""".strip()


prompt = build_rag_prompt(question, context)
print(prompt)

Answer the question using ONLY the context provided below.

Rules:
- Do not invent facts.
- If the answer is not present in the context, say:
  "I could not find this information in the provided documents."
- Keep the answer concise and factual.
- Mention the relevant source when possible.

Context:
[Source: accessibility_manual.txt]
Warranty:
The device is covered by a 24-month limited warranty for manufacturing defects and hardware failure under normal use.
Returns are accepted within 30 days with proof of purchase.

[Source: accessibility_manual.txt]
Accessibility Support Guidelines

The ACCESSIBILITY-2024 device supports customers with visual, auditory, motor, and cognitive accessibility requirements.

Environment Requirements:
Maintain an operating temperature between 0°C and 40°C and humidity between 10% and 90% non-condensing.
The device can connect through Wi-Fi or Ethernet.

Performance Optimization:
Use wired Ethernet for critical services when possible. Optimize Wi-Fi channe

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL_NAME
)

model = model.to(DEVICE)

print("LLM loaded successfully.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded successfully.


In [13]:
def generate_answer(prompt, max_new_tokens=200):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad(): #"I'm only asking the model to generate an answer. Don't calculate gradients.This saves memory and computation.
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=False
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

answer = generate_answer(prompt)
print(answer)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


24-month


In [14]:
def rag(question: str, top_k: int = TOP_K):
    retrieved = retrieve(question, top_k=top_k)
    context = build_context(retrieved)
    prompt = build_rag_prompt(question, context)
    answer = generate_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "retrieved": retrieved,
        "context": context,
        "prompt": prompt,
    }


result = rag("What are the recommended performance optimization practices?")

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVED SOURCES:")
display(result["retrieved"][["rank", "distance", "metadata"]])

QUESTION:
What are the recommended performance optimization practices?

ANSWER:
Use wired Ethernet for critical services when possible. Optimize Wi-Fi channels to reduce interference, use quality-of-service settings where appropriate, and limit unnecessary background network usage.

RETRIEVED SOURCES:


,rank,distance,metadata
0,1,1.728902,"{'page': '', 'source': 'account_management_man..."
1,2,1.760431,"{'doc_id': 'accessibility_manual', 'chunk_inde..."
2,3,2.057069,"{'page': '', 'doc_id': 'accessibility_manual',..."


In [16]:
questions = [
    "What are the environment requirements?",
    "How long is the warranty?",
    "How can performance be optimized?",
    "What authentication methods are supported?",
    "What should an administrator check during a login failure?",
    "What is the moon's average distance from Earth?"
]

for q in questions:
    print("=" * 90)
    print("QUESTION:", q)

    result = rag(q, top_k=TOP_K)

    print("ANSWER:", result["answer"])
    print("SOURCES:")
    for _, row in result["retrieved"].iterrows():
        print(" -", row["metadata"])
    print()

QUESTION: What are the environment requirements?
ANSWER: Maintain an operating temperature between 0°C and 40°C and humidity between 10% and 90% non-condensing.
SOURCES:
 - {'doc_id': 'accessibility_manual', 'page': '', 'chunk_index': 0, 'source': 'accessibility_manual.txt'}
 - {'doc_id': 'account_management_manual', 'source': 'account_management_manual.txt', 'chunk_index': 0, 'page': ''}
 - {'chunk_index': 1, 'source': 'accessibility_manual.txt', 'doc_id': 'accessibility_manual', 'page': ''}

QUESTION: How long is the warranty?
ANSWER: 24-month
SOURCES:
 - {'source': 'accessibility_manual.txt', 'page': '', 'chunk_index': 1, 'doc_id': 'accessibility_manual'}
 - {'page': '', 'source': 'accessibility_manual.txt', 'chunk_index': 0, 'doc_id': 'accessibility_manual'}
 - {'page': '', 'chunk_index': 0, 'doc_id': 'account_management_manual', 'source': 'account_management_manual.txt'}

QUESTION: How can performance be optimized?
ANSWER: Use wired Ethernet for critical services when possible. Op